# 00 — CPU data check

Offline, CPU-only, zero-cost. Runs the same code as `scripts/inspect_sources.py`
and `scripts/prepare_data.py` against the synthetic test fixture by default.
No model calls, no GPU, no network. See `docs/kaggle_quickstart.md` for the
GPU notebook (`01_kaggle_text_pilot.ipynb`) -- that one is gated and separate.

Cells are thin wrappers: all real logic lives in `src/urop/` and `scripts/`,
this notebook just calls it and prints results.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Could not locate repo root (no pyproject.toml found in any parent).")

REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "src"))
print("Repo root:", REPO_ROOT)

## 1. Source audit

What reference material and vendored upstream code is actually present, with hashes.

In [ ]:
import json
import subprocess

result = subprocess.run(
    [sys.executable, str(REPO_ROOT / "scripts" / "inspect_sources.py")],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)

## 2. Load, validate, clean, split

Default input is the synthetic fixture (`tests/fixtures/tiny_trajectories.jsonl`). Pass a real `--input`/`--domain` once real trajectory data exists under `data/` (git-ignored).

In [ ]:
from urop.data import load_and_clean_jsonl
from urop.splits import make_split, verify_no_overlap, assign_split

INPUT_PATH = REPO_ROOT / "tests" / "fixtures" / "tiny_trajectories.jsonl"
DOMAIN = "synthetic"

records, report = load_and_clean_jsonl(INPUT_PATH, domain=DOMAIN)
print(f"accepted={report.accepted} rejected={report.rejected} duplicates={report.duplicate_composite_ids} "
      f"of {report.total_lines} lines")
for err in report.errors:
    print(" -", err)

In [ ]:
manifest = make_split(records, seed=42, train_frac=0.8)
verify_no_overlap(manifest)
assigned = assign_split(records, manifest)
print(f"train tasks: {len(manifest.train_task_keys)}  test tasks: {len(manifest.test_task_keys)}")
print(f"train records: {len(assigned['train'])}  test records: {len(assigned['test'])}")

## 3. Budgeted mock coding dry run

Exercises the same budget/resume/manifest machinery Phase 1/2 will use for real, entirely against `MockCoder` (never a real model).

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    result = subprocess.run(
        [sys.executable, str(REPO_ROOT / "scripts" / "run_text_pilot.py"),
         "--config", str(REPO_ROOT / "configs" / "text_smoke.yaml"),
         "--output-dir", tmp],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr, file=sys.stderr)
    summary = subprocess.run(
        [sys.executable, str(REPO_ROOT / "scripts" / "summarize_run.py"), "--run-dir", tmp],
        capture_output=True, text=True,
    )
    print(summary.stdout)

## 4. Run the full pytest suite

Same tests as CI/local `pytest tests/` -- run here too so this notebook is a single place to confirm the CPU foundation is healthy.

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", str(REPO_ROOT / "tests"), "-q"],
                         capture_output=True, text=True, cwd=REPO_ROOT)
print(result.stdout[-3000:])
print(result.stderr[-2000:])
assert result.returncode == 0, "pytest failed -- see output above"
print("All CPU tests passed.")